# 04 Scanpy pseudobulk PCA / NMF review

Scanpy 기반 pseudobulk PCA와 ComBat-seq NMF 결과를 검토하고,
선택한 PC 수 / NMF module 목록을 JSON으로 저장하는 notebook입니다.

In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from subtype.io_utils import load_run_outputs

In [ ]:
RUN_DIR = PROJECT_ROOT / 'data' / '20260309_pilot' / 'results' / 'subtype_discovery' / 'base_run'
run = load_run_outputs(RUN_DIR)
RUN_DIR

## PCA variance table

In [ ]:
run['pca_explained_variance'].head(20) if run['pca_explained_variance'] is not None else pd.DataFrame()

## Scree plots

In [ ]:
scree_dir = RUN_DIR / 'pca_scree_plots'
for png in sorted(scree_dir.glob('*.png')):
    print(png.name)
    display(Image(filename=str(png)))

## Save per-celltype PC selection

아래 사전을 수정한 뒤 셀을 실행하면 JSON이 저장됩니다.
이 파일을 config의 `pca_features.pc_selection_file`에 연결해서 사용하면 됩니다.

In [ ]:
PC_SELECTION = {
    'naive/TCM CD4+T': 3,
    'CD14 Mono': 4,
    'effector CD8+T': 3,
    'NK': 3,
}

pc_selection_path = RUN_DIR / 'pca_pc_selection.json'
pc_selection_path.write_text(json.dumps(PC_SELECTION, indent=2, ensure_ascii=False), encoding='utf-8')
pc_selection_path

## NMF module score / gene list review

In [ ]:
run['nmf_module_scores'].head() if run['nmf_module_scores'] is not None else pd.DataFrame()

In [ ]:
run['nmf_gene_lists'].head(50) if run['nmf_gene_lists'] is not None else pd.DataFrame()

In [ ]:
heatmap_dir = RUN_DIR / 'nmf_review' / 'heatmaps'
for png in sorted(heatmap_dir.glob('*.png')):
    print(png.name)
    display(Image(filename=str(png)))

## Save selected NMF modules for downstream use

biology가 해석되는 module만 아래 리스트에 넣고 저장하면 됩니다.
이 파일을 config의 `nmf_features.selected_modules_file`에 연결하세요.

In [ ]:
SELECTED_MODULES = [
    # 'CD14 Mono_NMF1',
    # 'NK_NMF2',
]

selected_modules_path = RUN_DIR / 'nmf_selected_modules.json'
selected_modules_path.write_text(
    json.dumps({'selected_modules': SELECTED_MODULES}, indent=2, ensure_ascii=False),
    encoding='utf-8',
)
selected_modules_path